# Fraud Detection ML Pipeline

This notebook walks through every step used to build the Random Forest fraud detection model deployed in this project.

**Steps covered:**
1. Import Libraries
2. Load & Explore Data
3. Data Cleaning
4. Feature Engineering
5. Encoding Categorical Features
6. Train / Test Split
7. Handle Class Imbalance (SMOTE)
8. Train Random Forest Model
9. Evaluate the Model
10. Save Model Artifacts

---
## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
import joblib

print('All libraries imported successfully.')

---
## Step 2 — Load & Explore Data

The dataset used is **`consolidated_fraud_data.csv`** which is based on IEEE-CIS Fraud Detection features.

Key columns:
- `TransactionID` — unique transaction identifier
- `TransactionAmt` — transaction amount in USD
- `card1` — payment card identifier
- `DeviceType` — desktop / mobile
- `P_emaildomain` — purchaser email domain
- `isFraud` — target label (1 = fraud, 0 = legit)

In [ ]:
df = pd.read_csv('backend/app/data/consolidated_fraud_data 3.csv')

print('Shape:', df.shape)
print('\nColumn types:')
print(df.dtypes.value_counts())
df.head()

In [ ]:
# Class distribution
fraud_counts = df['isFraud'].value_counts()
print('Class distribution:')
print(fraud_counts)
print(f'\nFraud rate: {fraud_counts[1] / len(df) * 100:.2f}%')

fraud_counts.plot(kind='bar', color=['steelblue', 'crimson'], title='Fraud vs Legit')
plt.xticks([0, 1], ['Legit', 'Fraud'], rotation=0)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Missing values overview
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).sort_values(ascending=False)
print('Top 20 columns with missing values:')
print(missing_pct[missing_pct > 0].head(20))

---
## Step 3 — Data Cleaning

- Drop columns with > 50% missing values (too sparse to be useful)
- Fill remaining numeric nulls with median
- Fill remaining categorical nulls with `'unknown'`

In [ ]:
# Drop columns with > 50% missing
threshold = 0.5
before = df.shape[1]
df = df.loc[:, df.isnull().mean() < threshold]
print(f'Dropped {before - df.shape[1]} columns with >50% missing. Remaining: {df.shape[1]}')

# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

# Fill nulls
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
df[cat_cols] = df[cat_cols].fillna('unknown')

print(f'Remaining nulls: {df.isnull().sum().sum()}')

---
## Step 4 — Feature Engineering

Create new features that help the model detect fraud patterns:
- `hour_of_day` — transactions at unusual hours are riskier
- `is_large_txn` — flag high-value transactions
- `amt_log` — log-transform skewed transaction amount

In [ ]:
# Log-transform TransactionAmt (reduces right skew)
df['amt_log'] = np.log1p(df['TransactionAmt'])

# Flag large transactions (> $500)
df['is_large_txn'] = (df['TransactionAmt'] > 500).astype(int)

# Distribution of transaction amount by fraud label
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.groupby('isFraud')['TransactionAmt'].hist(bins=50, ax=axes[0], alpha=0.7)
axes[0].set_title('Raw Transaction Amount')
axes[0].legend(['Legit', 'Fraud'])

df.groupby('isFraud')['amt_log'].hist(bins=50, ax=axes[1], alpha=0.7)
axes[1].set_title('Log Transaction Amount')
axes[1].legend(['Legit', 'Fraud'])
plt.tight_layout()
plt.show()

---
## Step 5 — Encode Categorical Features

Random Forest cannot handle strings — we use **LabelEncoder** on each categorical column and save the encoders so the API can use them at inference time.

In [ ]:
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

print(f'Encoded {len(cat_cols)} categorical columns.')
print('Sample:', cat_cols[:5])

---
## Step 6 — Train / Test Split

Split **80% train / 20% test**, stratified on the fraud label so both splits have the same fraud rate.

In [ ]:
# Drop ID columns that leak or have no predictive value
drop_cols = ['TransactionID', 'isFraud']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols]
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {X_train.shape}')
print(f'Test size  : {X_test.shape}')
print(f'Train fraud rate: {y_train.mean()*100:.2f}%')
print(f'Test  fraud rate: {y_test.mean()*100:.2f}%')

---
## Step 7 — Handle Class Imbalance with SMOTE

Fraud datasets are heavily imbalanced (typically < 4% fraud). Training directly produces a model biased toward predicting everything as legit.

**SMOTE** (Synthetic Minority Over-sampling Technique) generates synthetic fraud samples in the training set only — never in the test set.

In [ ]:
print('Before SMOTE:', y_train.value_counts().to_dict())

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('After SMOTE :', pd.Series(y_train_res).value_counts().to_dict())

# Visual check
pd.Series(y_train_res).value_counts().plot(
    kind='bar', color=['steelblue', 'crimson'],
    title='Training Set After SMOTE'
)
plt.xticks([0, 1], ['Legit', 'Fraud'], rotation=0)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

---
## Step 8 — Train Random Forest Model

Random Forest is an ensemble of decision trees. Key hyperparameters:
- `n_estimators=200` — 200 trees for stable predictions
- `max_depth=20` — limits overfitting
- `class_weight='balanced'` — extra weight on minority (fraud) class
- `n_jobs=-1` — use all CPU cores

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_res, y_train_res)
print('Model training complete.')

---
## Step 9 — Evaluate the Model

We use multiple metrics because accuracy is misleading on imbalanced data:
- **Precision** — of all flagged fraud, how many were real?
- **Recall** — of all real fraud, how many did we catch?
- **ROC-AUC** — overall discrimination ability
- **PR-AUC** — better metric for imbalanced classes

In [ ]:
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_proba):.4f}')
print(f'PR-AUC  : {average_precision_score(y_test, y_proba):.4f}')

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Legit', 'Fraud'],
    ax=axes[0], colorbar=False
)
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='darkorange', lw=2,
             label=f'ROC AUC = {roc_auc_score(y_test, y_proba):.3f}')
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_proba)
axes[2].plot(recall, precision, color='steelblue', lw=2,
             label=f'PR AUC = {average_precision_score(y_test, y_proba):.3f}')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve')
axes[2].legend()

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=120)
plt.show()

In [ ]:
# Feature Importance — top 20
importances = pd.Series(rf.feature_importances_, index=feature_cols)
top20 = importances.nlargest(20)

plt.figure(figsize=(10, 6))
top20.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 20 Feature Importances')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120)
plt.show()

print('\nTop 10 features:')
print(top20.head(10))

In [ ]:
# Fraud score distribution
plt.figure(figsize=(10, 4))
plt.hist(y_proba[y_test == 0], bins=50, alpha=0.6, label='Legit', color='steelblue')
plt.hist(y_proba[y_test == 1], bins=50, alpha=0.6, label='Fraud', color='crimson')
plt.axvline(x=0.5, color='black', linestyle='--', label='Threshold 0.5')
plt.xlabel('Fraud Probability Score')
plt.ylabel('Count')
plt.title('Score Distribution by True Label')
plt.legend()
plt.tight_layout()
plt.savefig('score_distribution.png', dpi=120)
plt.show()

---
## Step 10 — Save Model Artifacts

Save the trained model and all encoders/scalers so the FastAPI backend can load them at startup without retraining.

Saved files:
- `random_forest_fraud_model.pkl` — the trained RF model
- `label_encoder.pkl` — dict of LabelEncoders per categorical column
- `scaler.pkl` — StandardScaler (if used)
- `metadata.json` — feature list and model version

In [ ]:
import json, os

os.makedirs('backend/app/ml', exist_ok=True)

# Save model
joblib.dump(rf, 'backend/app/ml/random_forest_fraud_model.pkl')
print('Model saved.')

# Save label encoders
joblib.dump(encoders, 'backend/app/ml/label_encoder.pkl')
print('Label encoders saved.')

# Save scaler (fit on training data)
scaler = StandardScaler()
scaler.fit(X_train_res)
joblib.dump(scaler, 'backend/app/ml/scaler.pkl')
print('Scaler saved.')

# Save metadata
metadata = {
    'model_type': 'RandomForestClassifier',
    'n_estimators': 200,
    'feature_names': list(feature_cols),
    'n_features': len(feature_cols),
    'roc_auc': round(roc_auc_score(y_test, y_proba), 4),
    'pr_auc': round(average_precision_score(y_test, y_proba), 4),
    'version': 'v1'
}
with open('model/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Metadata saved.')
print('\nAll artifacts saved successfully!')
print(json.dumps(metadata, indent=2))

---
## Summary

| Step | What we did |
|------|-------------|
| 1 | Imported all required libraries |
| 2 | Loaded dataset, explored shape, class balance, missing values |
| 3 | Dropped sparse columns, imputed remaining nulls |
| 4 | Engineered `amt_log`, `is_large_txn` features |
| 5 | Label-encoded all categorical columns, saved encoders |
| 6 | Stratified 80/20 train-test split |
| 7 | Applied SMOTE on training set to balance fraud/legit |
| 8 | Trained Random Forest (200 trees, balanced weights) |
| 9 | Evaluated with confusion matrix, ROC-AUC, PR-AUC, feature importance |
| 10 | Saved model + encoders + scaler + metadata for API use |

The saved model is loaded by `backend/app/ml/model_loader.py` and called via `backend/app/ml/predictor.py` for real-time inference.